<a href="https://colab.research.google.com/github/hiiamlars/master_thesis/blob/main/tvd_mi.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Setup

In [3]:
# @title Dependencies

# Installations of third-party libraries
!pip install openai -q
!pip install anthropic -q
!pip install together -q
!pip install tqdm -q

# First-party imports
import os
import json
import csv
import time
import random
from typing import List, Dict, Any, Optional, Tuple
import numpy as np # Import numpy

# Third-party imports
import pandas as pd
from openai import OpenAI
from anthropic import Anthropic
from together import Together
from tqdm.notebook import tqdm
from google.colab import drive

print("\nInstalling and loading dependencies complete!")


Installing and loading dependencies complete!


In [4]:
# @title Paths

# Mount GDrive
drive.mount('/content/drive')

# Set working directory
WORKING_DIR = "/content/drive/MyDrive/Thesis/"

# Create working directory (if not already existing)
os.makedirs(WORKING_DIR, exist_ok=True)
print(f"Ensured working directory exists at: {WORKING_DIR}.")

# Define and create PAPER_DIR for paper content
PAPER_DIR = os.path.join(WORKING_DIR, "iclr/final")
os.makedirs(PAPER_DIR, exist_ok=True)
print(f"Ensured raw directory exists at: {PAPER_DIR}.")

# Define and create LLM_DIR for llm based reviews
LLM_DIR = os.path.join(WORKING_DIR, "llm_reviews")
os.makedirs(LLM_DIR, exist_ok=True)
print(f"Ensured llm review directory exists at: {LLM_DIR}.")

# Define and create RES_DIR for the result file
RES_DIR = os.path.join(WORKING_DIR, "tvd_mi")
os.makedirs(RES_DIR, exist_ok=True)
print(f"Ensured result directory exists at: {RES_DIR}.")

# Define and create RES_DIR for the log file
LOG_DIR = os.path.join(WORKING_DIR, "tvd_mi")
os.makedirs(LOG_DIR, exist_ok=True)
print(f"Ensured logging directory exists at: {LOG_DIR}.")

Mounted at /content/drive
Ensured working directory exists at: /content/drive/MyDrive/Thesis/.
Ensured raw directory exists at: /content/drive/MyDrive/Thesis/iclr/final.
Ensured llm review directory exists at: /content/drive/MyDrive/Thesis/llm_reviews.
Ensured result directory exists at: /content/drive/MyDrive/Thesis/tvd_mi.
Ensured logging directory exists at: /content/drive/MyDrive/Thesis/tvd_mi.


In [5]:
# @title Definitions

# Active model
MODEL = "gpt-4o-mini" ###! The authors hade this one activ in their repo code but I could not find any other mention which model used here. Can/Should I delete the others?
# MODEL = "claude-3-5-haiku-20241022"
# MODEL = "meta-llama/Llama-3.3-70B-Instruct-Turbo"

# Delay for retry
TIMEOUT_SECONDS = 1200

# API Keys
OPENAI_KEY = "OPENAI_KEY_PLACEHOLDER"
ANTHROPIC_KEY = "ANTHROPIC_KEY_PLACEHOLDER"
TOGETHER_KEY = "TOGETHER_KEY_PLACEHOLDER"

# Define the type of TVD-MI score to calculate using a numerical option
# 1: "paper / human review"
# 2: "paper / llm review"
# 3: "human review / llm review"
# 4: "llm review / llm review"
TVD_MI_SCORE_OPTION = 2

# Define parameter columns for grouping and logging
PARAM_COLS = ["model", "temperature", "reasoning_effort", "chain_of_thought", "scope"]

# Define log file columns
LOG_COL = ['combo', 'TPR_calculated', 'FPR_calculated', 'complete']

# Define col names for the result file for combined TPR and FPR on one row
RES_COL = [
    'paper_id',
] + PARAM_COLS + [
    'tpr_response_a', 'tpr_response_b', 'tpr_tvd_mi_score_1', 'tpr_tvd_mi_score_2', 'tpr_avg_tvd_mi_score',
    'fpr_response_a', 'fpr_response_b', 'fpr_tvd_mi_score_1', 'fpr_tvd_mi_score_2', 'fpr_avg_tvd_mi_score'
]

SEED_VALUE = 7 # Random value

SIMULATION_ACTIVE = True # SIMULATION_ACTIVE = False activates the API calls

print("\nDefintions complete!")


Defintions complete!


In [6]:
# @title Load data

# Read dataset_paper
dataset_paper = pd.read_parquet(os.path.join(PAPER_DIR, "dataset_paper.parquet"))

# Select 'paper_id' and the target columns for paper content and human reviews
paper_content = dataset_paper[['paper_id', 'parsed_text', 'human_review_1', 'human_review_2', 'human_review_3']] # change to human_review_1 etc.

# Check paper_content
print(paper_content.head(3))

# Read llm reviews
dataset_llm = pd.read_parquet(os.path.join(LLM_DIR, "llm_sim_results.parquet"))

# Check dataset_llm
print(dataset_llm.head(3))

print("\nData loaded!")

      paper_id                                        parsed_text  \
0  vuD2xEtxZcj  MINIMUM VARIANCE UNBIASED N:M SPARSITY FOR THE...   
1  WoByU5W5te0  GECONERF: FEW-SHOT NEURAL RADIANCE FIELDS VIA ...   
2  XZRmNjUMj0c  EFFICIENT ONE-SHOT NEURAL ARCHITECTURE SEARCH ...   

                                      human_review_1  \
0  Summary Of The Paper:\n\nThe paper aims to acc...   
1  Summary Of The Paper:\n\nThis paper proposes a...   
2  Summary Of The Paper:\n\nThis paper proposes a...   

                                      human_review_2  \
0  Summary Of The Paper:\n\nThis paper develops a...   
1  Summary Of The Paper:\n\nThis paper focus on t...   
2  Summary Of The Paper:\n\nThis paper focused on...   

                                      human_review_3  
0  Summary Of The Paper:\n\nThe paper presents a ...  
1  Summary Of The Paper:\n\nThe paper proposes a ...  
2  Summary Of The Paper:\n\nThe paper introduces ...  
      paper_id                  model temperature rea

In [7]:
# @title Merge data

# Merge paper content with LLM data
merged_df = pd.merge(paper_content, dataset_llm, on='paper_id', how='outer')
print(merged_df.head(3))
print(merged_df.shape)

print("\nData merged!")

      paper_id                                        parsed_text  \
0  -Y34L45JR6z  POLICY EXPANSION FOR BRIDGING OFFLINE-TO-ONLIN...   
1  -Y34L45JR6z  POLICY EXPANSION FOR BRIDGING OFFLINE-TO-ONLIN...   
2  -Y34L45JR6z  POLICY EXPANSION FOR BRIDGING OFFLINE-TO-ONLIN...   

                                      human_review_1  \
0  Summary Of The Paper:\n\nThe paper addresses h...   
1  Summary Of The Paper:\n\nThe paper addresses h...   
2  Summary Of The Paper:\n\nThe paper addresses h...   

                                      human_review_2  \
0  Summary Of The Paper:\n\nThis paper presents a...   
1  Summary Of The Paper:\n\nThis paper presents a...   
2  Summary Of The Paper:\n\nThis paper presents a...   

                                      human_review_3                  model  \
0  Summary Of The Paper:\n\nThe paper considers t...  gpt-5-mini-2025-08-07   
1  Summary Of The Paper:\n\nThe paper considers t...  gpt-5-mini-2025-08-07   
2  Summary Of The Paper:\n\nThe pape

### API

In [8]:
class LLMClient:
    """
    Unified client handling Simulation, Caching, and Routing
    for OpenAI, Anthropic, and Together.
    """
    def __init__(self, simulate: bool = True, seed: int = 7):
        self.simulate = simulate
        self.rng = random.Random(seed)
        self.cache = {}
        self._openai = None
        self._anthropic = None
        self._together = None

    def _maybe_init_clients(self):
        """
        Initialize clients if simulation is off
        """
        if self.simulate:
            return

        if self._openai is None and "OPENAI_KEY" in globals():
            self._openai = OpenAI(api_key=OPENAI_KEY)
        if self._anthropic is None and "ANTHROPIC_KEY" in globals():
            self._anthropic = Anthropic(api_key=ANTHROPIC_KEY)
        if self._together is None and "TOGETHER_KEY" in globals():
            self._together = Together(api_key=TOGETHER_KEY)

    def call(
        self,
        model: str,
        prompt: str,
        temperature: float = 0.0,
        max_tokens: int = 500,
        max_retries: int = 5
    ) -> Tuple[str, Dict[str, Any]]:
        """
        Call the API / simulate with the given parameters
        """
        self._maybe_init_clients()

        # Simulation
        if self.simulate:

            # Create artificial response
            possible_responses = [
                "[significant gain]",
                "[little gain]",
                "[no gain]",
                "[total failure]"
            ]
            simulated_response = self.rng.choice(possible_responses)

            raw = {
                "model": model,
                "temperature": temperature,
                "max_tokens": max_tokens,
                "provider": "simulated",
                "response": simulated_response
            }

            # Make the response human readable
            structured = json.dumps(raw, indent=2)

            return structured, raw

        # APIs
        for attempt in range(max_retries):
            try:

                # Create result items
                response_text = ""
                provider = ""

                # ANTHROPIC
                if "claude" in model.lower():
                    resp = self._anthropic.messages.create(
                        model=model,
                        max_tokens=max_tokens,
                        temperature=temperature,
                        messages=[{"role": "user", "content": prompt}]
                    )
                    response_text = resp.content[0].text
                    provider = "anthropic"

                # OPENAI
                elif "gpt" in model.lower():
                    resp = self._openai.chat.completions.create(
                        model=model,
                        messages=[{"role": "user", "content": prompt}],
                        temperature=temperature,
                        max_tokens=max_tokens
                    )
                    response_text = resp.choices[0].message.content
                    provider = "openai"

                # TOGETHER
                else:
                    resp = self._together.chat.completions.create(
                        model=model,
                        messages=[{"role": "user", "content": prompt}],
                        temperature=temperature,
                        max_tokens=max_tokens
                    )
                    response_text = resp.choices[0].message.content
                    provider = "together"

                # Create full response
                raw = {
                    "model": model,
                    "temperature": temperature,
                    "max_tokens": max_tokens,
                    "provider": provider,
                    "response": response_text
                }

                # Make the response human readable
                structured = json.dumps(raw, indent=2)

                return structured, raw

            # Run again after delay until max_retries
            except Exception as e:
                wait_time = (2 ** attempt) + random.random()
                print(f"Error on attempt {attempt+1}: {e}. Retrying in {wait_time:.2f}s...")
                time.sleep(wait_time)

        # Create error in case of failure till max_retries
        error_raw = {"status": "error", "model": model, "response": "[total failure]"}

        # Make error human readable
        return json.dumps(error_raw, indent=2), error_raw

def generate_tvd_mi_prompt(response_a: str, response_b: str) -> str:
    """Generate prompt for TVD-MI critic"""
    prompt = f"""You are evaluating whether two responses could be from agents working on the same task.

Task description: Scientific paper peer review task

Response A: {response_a}

Response B: {response_b}

Do these responses show evidence of coming from the same task/source? Consider:
- Shared specific details, facts, or entities
- Similar topics or themes
- Overlapping information that would be unlikely if from different sources

Respond in two sentences or less. Put the information gain at the end in brackets in this format:

Rate the information gain:
- [[Significant Gain]]: Clear evidence they're from the same source (many shared specifics)
- [[Little Gain]]: Some shared elements but also differences
- [[No Gain]]: No evidence of shared source (could be from completely different tasks)""" # Should I adapt the first sentence to my own purpose?

    return prompt

def interpret_tvd_mi_response(response: str) -> float:
    """Convert LLM response to numeric score"""
    response = response.strip().lower()

    if "[significant gain]" in response:
        return 1.0
    elif "[little gain]" in response:
        return 0.25
    elif "[no gain]" in response:
        return 0.0
    elif "[total failure]" in response:
        return np.nan
    else:
        # Default to no gain if response is unclear
        print(f"Warning: Unclear response '{response}', defaulting to [[No Gain]]")
        return 0.0

def _generate_notes_if_paper_content(llm_client, model, content_text: str, content_source_column: str) -> Tuple[str, bool]:
    """
    Generates notes from paper content (parsed_text or abstract) if the source column matches.
    If note generation fails (TOTAL FAILURE from LLM), returns the original `content_text` and False.
    If the source column is not what is expected, raises a ValueError.
    """
    # Only attempt note generation if the content is from 'parsed_text' or 'abstract'
    if content_source_column in ['parsed_text', 'abstract']:
        notes_prompt = f"Take notes on the paper for an ICLR style review\n\nPaper Content: {content_text}"
        _, raw_notes_response = llm_client.call(
            model=model,
            prompt=notes_prompt,
            temperature=0.0,
            max_tokens=1000 # Allow for longer notes
        )
        generated_notes = raw_notes_response.get('response', '')

        # If note generation failed (returned TOTAL FAILURE), return original content_text and False
        if generated_notes == "[TOTAL FAILURE]":
            return content_text, False # Return original content, signal note generation failure ###! Or should I treat this as it is? The authors make no claim about this case.

        # Otherwise, return the generated notes and True
        return generated_notes, True
    # If the content_cource_column is not what is expected
    else:
      raise ValueError(f"Unexpected content_source_column: {content_source_column}")

### General helpers

In [9]:
# Helper to handle NaN values when creating hashable keys for comparison
def _nan_to_none(x):
    try:
        if pd.isna(x):
            return None
    except Exception:
        pass
    return x

# Function to create a unique hashable key for a paper-parameter combination
def get_unique_combo_key(row, param_cols):
    key_elements = [row['paper_id']]
    for col in param_cols:
        key_elements.append(_nan_to_none(row[col]))
    return tuple(key_elements)

# Helper to format parameter string for logging and printing
def format_param_strings(paper_id_val, param_dict):
    # For log file: paper_id=X|model=Y|temp=Z...
    log_parts = [f"paper_id={paper_id_val}"]
    for col in PARAM_COLS:
        value = param_dict.get(col)
        log_parts.append(f"{col}={_nan_to_none(value)}")
    combo_str = "|".join(log_parts)

    # For print message: model=Y, temperature=Z...
    # Note: paper_id will be prefixed separately for printing
    print_param_parts = []
    for col in PARAM_COLS:
        value = param_dict.get(col)
        print_param_parts.append(f"{col}={'' if pd.isna(value) else value}")
    print_params_str = ", ".join(print_param_parts)

    return combo_str, print_params_str

# Helper to parse values from the string representation in combo
def _parse_val_from_log_str(val_str):
    if val_str == 'None':
        return None
    try:
        # Attempt float conversion for numbers (e.g., temperature)
        return float(val_str)
    except ValueError:
        return val_str # If not a float, return as string (e.g., 'low', 'abstract')

# Function to create a unique hashable key from the combo string for log file parsing
def get_unique_combo_key_from_log_entry(combo_str_from_log, param_cols):
    key_elements = []
    parts = combo_str_from_log.split('|')

    # Extract paper_id from the first part
    paper_id_part = parts[0]
    paper_id = paper_id_part.split('=', 1)[1]
    key_elements.append(paper_id)

    # Extract parameter values from the remaining parts
    param_map = {}
    for part in parts[1:]:
        key, val_str = part.split('=', 1)
        param_map[key] = _parse_val_from_log_str(val_str)

    # Assemble key_elements in the same order as get_unique_combo_key expects
    for col in param_cols:
        key_elements.append(param_map.get(col)) # Use .get to safely handle cases where a param might be missing

    return tuple(key_elements)

print("\nFunctions defined!")


Functions defined!


In [10]:
def configure_tvd_mi_run(option: int, res_dir: str, log_dir: str) -> Tuple[List[str], str, str]:
    """
    Configures the column pair and output file names based on the TVD-MI score option.
    Randomly selects human/LLM review columns if multiple are available.
    """
    column_pair = []
    file_suffix = ""
    score_type_label = ""

    # Dynamically get all human review columns
    all_human_review_cols = [col for col in paper_content.columns if col.startswith('human_review_')]
    # Dynamically get all LLM review columns
    all_llm_review_cols = [col for col in dataset_llm.columns if col.startswith('llm_review_')]

    # Randomly select one human review column if available
    selected_human_review_col = random.choice(all_human_review_cols) if all_human_review_cols else ""
    # Randomly select one LLM review column if available
    selected_llm_review_col = random.choice(all_llm_review_cols) if all_llm_review_cols else ""

    if option == 1:
        score_type_label = "paper / human review"
        if not selected_human_review_col:
            raise ValueError("No human review columns available for 'paper / human review' comparison.")
        column_pair = ["parsed_text", selected_human_review_col]
        file_suffix = f"paper_{selected_human_review_col}"

    elif option == 2:
        score_type_label = "paper / llm review"
        if not selected_llm_review_col:
            raise ValueError("No LLM review columns available for 'paper / llm review' comparison.")
        column_pair = ["parsed_text", selected_llm_review_col]
        file_suffix = f"paper_{selected_llm_review_col}"

    elif option == 3:
        score_type_label = "human review / llm review"
        if not selected_human_review_col or not selected_llm_review_col:
            raise ValueError("Not enough human/LLM review columns available for 'human review / llm review' comparison.")
        column_pair = [selected_human_review_col, selected_llm_review_col]
        file_suffix = f"h_{selected_human_review_col}_l_{selected_llm_review_col}"

    elif option == 4:
        score_type_label = "llm review / llm review"
        # Check if there are sufficient llm reviews for this case
        if len(all_llm_review_cols) < 2:
            raise ValueError("Not enough LLM review columns available for 'llm review / llm review' comparison (need at least two distinct ones).")

        # Select the first LLM review column
        selected_llm_review_col_1 = selected_llm_review_col

        # Select a second, *different* LLM review column
        remaining_llm_review_cols = [col for col in all_llm_review_cols if col != selected_llm_review_col_1]
        selected_llm_review_col_2 = random.choice(remaining_llm_review_cols)

        column_pair = [selected_llm_review_col_1, selected_llm_review_col_2]
        file_suffix = f"llm_{selected_llm_review_col_1}_llm_{selected_llm_review_col_2}"
    else:
        raise ValueError(f"Invalid TVD_MI_OPTION: {option}. Please choose between 1 and 4.")

    # Create output and log paths
    output_path = os.path.join(res_dir, f"tvd_mi_scores_{file_suffix}.parquet")
    log_path = os.path.join(log_dir, f"tvd_mi_log_{file_suffix}.parquet")


    return column_pair, output_path, log_path

### Higher order Functions

In [11]:
def calculate_tvd_mi_scores_for_pair(llm_client, model, response_a_text, response_b_text):
    """
    Generates prompts, calls LLM, and interprets scores for a given pair of responses.
    """
    # --- A, B ---
    # Generate the TVD-MI prompt
    tvd_mi_prompt_1 = generate_tvd_mi_prompt(response_a_text, response_b_text)

    # Call the LLMClient
    structured_response_1, raw_response_1 = llm_client.call(
        model=model,
        prompt=tvd_mi_prompt_1,
        temperature=0.0,
        max_tokens=500
    )

    # Interpret the LLM's response to get the TVD-MI score
    llm_critic_response_text_1 = raw_response_1.get('response')
    tvd_mi_score_1 = interpret_tvd_mi_response(llm_critic_response_text_1)

    # --- B, A ---
    # Generate the TVD-MI prompt
    tvd_mi_prompt_2 = generate_tvd_mi_prompt(response_b_text, response_a_text)

    # Call the LLMClient
    structured_response_2, raw_response_2 = llm_client.call(
        model=model,
        prompt=tvd_mi_prompt_2,
        temperature=0.0,
        max_tokens=500
    )

    # Interpret the LLM's response to get the TVD-MI score
    llm_critic_response_text_2 = raw_response_2.get('response')
    tvd_mi_score_2 = interpret_tvd_mi_response(llm_critic_response_text_2)

    # --- Results ---
    # Calculate the average TVD-MI score
    avg_tvd_mi_score = (tvd_mi_score_1 + tvd_mi_score_2) / 2

    return {
        'response_a': response_a_text,
        'response_b': response_b_text,
        'prompt_1': tvd_mi_prompt_1,
        'raw_response_1': json.dumps(raw_response_1),
        'structured_response_1': structured_response_1,
        'prompt_2': tvd_mi_prompt_2,
        'raw_response_2': json.dumps(raw_response_2),
        'structured_response_2': structured_response_2,
        'tvd_mi_score_1': tvd_mi_score_1,
        'tvd_mi_score_2': tvd_mi_score_2,
        'avg_tvd_mi_score': avg_tvd_mi_score
    }

In [12]:
def process_single_paper(llm_client, model, paper_id, row, response_a_col_name, response_b_col_name, merged_df, param_cols):
    """
    Processes a single paper_id, calculating TPR and FPR scores and returning them.
    Returns a tuple of (results_data_dict, log_data_dict).
    """
    # --- Initialize results for single row in result file ---
    data_to_write = {'paper_id': paper_id}

    # Add parameter values from the current row to the results dictionary
    param_dict = {}
    for param_col in param_cols:
        param_dict[param_col] = row[param_col] # Store parameter values for this row
        data_to_write[param_col] = row[param_col]

    # --- TPR Calculation: response_a and response_b from the current row ---
    response_a_tpr_text = str(row[response_a_col_name]) if pd.notna(row[response_a_col_name]) else ""
    response_b_tpr_text = str(row[response_b_col_name]) if pd.notna(row[response_b_col_name]) else ""

    tpr_results_raw = calculate_tvd_mi_scores_for_pair(llm_client, model, response_a_tpr_text, response_b_tpr_text)

    # Determine if TPR calculation was successful
    tpr_calculated = not pd.isna(tpr_results_raw['avg_tvd_mi_score'])

    # Add TPR results with 'tpr_' prefix
    data_to_write['tpr_response_a'] = tpr_results_raw['response_a']
    data_to_write['tpr_response_b'] = tpr_results_raw['response_b']
    data_to_write['tpr_tvd_mi_score_1'] = tpr_results_raw['tvd_mi_score_1']
    data_to_write['tpr_tvd_mi_score_2'] = tpr_results_raw['tvd_mi_score_2']
    data_to_write['tpr_avg_tvd_mi_score'] = tpr_results_raw['avg_tvd_mi_score']

    # --- FPR Calculation: response_b from current row, response_a from random different row ---
    response_b_fpr_text = str(row[response_b_col_name]) if pd.notna(row[response_b_col_name]) else ""

    # Get all possible indices from the full merged_df, excluding the current row's index 'i'
    eligible_indices_for_a = merged_df.index.drop(row.name).tolist()

    if not eligible_indices_for_a:
        response_a_fpr_text = ""
    else:
        random_a_idx = random.choice(eligible_indices_for_a)
        response_a_fpr_text = str(merged_df.loc[random_a_idx, response_a_col_name]) if pd.notna(merged_df.loc[random_a_idx, response_a_col_name]) else ""

    fpr_results_raw = calculate_tvd_mi_scores_for_pair(llm_client, model, response_a_fpr_text, response_b_fpr_text)

    # Determine if FPR calculation was successful
    fpr_calculated = not pd.isna(fpr_results_raw['avg_tvd_mi_score'])

    # Add FPR results with 'fpr_' prefix
    data_to_write['fpr_response_a'] = fpr_results_raw['response_a']
    data_to_write['fpr_response_b'] = fpr_results_raw['response_b']
    data_to_write['fpr_tvd_mi_score_1'] = fpr_results_raw['tvd_mi_score_1']
    data_to_write['fpr_tvd_mi_score_2'] = fpr_results_raw['tvd_mi_score_2']
    data_to_write['fpr_avg_tvd_mi_score'] = fpr_results_raw['avg_tvd_mi_score']

    # --- Logging ---

    # Log data to return
    log_data_to_write = {
        'TPR_calculated': tpr_calculated,
        'FPR_calculated': fpr_calculated,
        'complete': tpr_calculated and fpr_calculated
    }

    # Format the combo string for logging
    combo_str, _ = format_param_strings(paper_id, param_dict)
    log_data_to_write['combo'] = combo_str

    return data_to_write, log_data_to_write

### TVD-MI score calculation

In [13]:
if __name__ == "__main__":

    # Set seed for reproducibility for the global random module
    random.seed(SEED_VALUE)

    # Activate/Deactivate simulation/API calls
    llm_client = LLMClient(simulate=SIMULATION_ACTIVE, seed=SEED_VALUE)

    # Configure the run based on the TVD_MI_SCORE_OPTION
    column_pair, result_path, log_path = configure_tvd_mi_run(TVD_MI_SCORE_OPTION, RES_DIR, LOG_DIR)

    # Select prompt input based on column_pair
    response_a_col_name = column_pair[0]
    response_b_col_name = column_pair[1]

    # Information output
    print(f"Prompt input will use columns {response_a_col_name} and {response_b_col_name}", flush = True)
    print(f"Processing and saving results to: {result_path}", flush = True)
    print(f"Logging successful iterations to: {log_path}", flush = True)

    # Initialize lists to store data for all papers and configurations
    all_results_data = []
    all_log_data = []

    # Build the "done" set from existing log file
    done_combinations = set()
    existing_log_df = pd.DataFrame(columns=LOG_COL) # Initialize empty DataFrame for log
    if os.path.exists(log_path):
        try:
            existing_log_df = pd.read_parquet(log_path)
            # Ensure 'complete' column is boolean for filtering
            if 'complete' in existing_log_df.columns:
                completed_log_df = existing_log_df[existing_log_df['complete'] == True]
                for _, log_row in completed_log_df.iterrows():
                    # Use the new helper function to parse the combo string
                    done_combinations.add(get_unique_combo_key_from_log_entry(log_row['combo'], PARAM_COLS))
                print(f"Loaded {len(done_combinations)} previously completed paper/parameter combinations from {log_path}. These will be skipped.", flush=True)
            else:
                print(f"Warning: 'complete' column not found in log file {log_path}. Cannot determine previously completed combinations.", flush=True)
        except Exception as e:
            print(f"Warning: Could not load or parse existing log file {log_path}. Error: {e}. Starting fresh for all combinations.", flush=True)

    # Initialize a counter for the global progress of processing paper-parameter combinations
    global_item_counter = 0
    total_items_to_process = len(merged_df) # Total combinations in merged_df

    # Group by parameter columns and then iterate through each group
    # Using sorted() to ensure consistent order of groups across runs
    grouped_df = merged_df.groupby(PARAM_COLS, dropna=False)

    print(f"Starting processing for {len(grouped_df)} parameter configuration groups.", flush=True)

    # Iterate through each parameter configuration group
    for group_idx, (params, group_df) in enumerate(tqdm(list(grouped_df), desc="Parameter Groups")):

        param_values_dict = dict(zip(PARAM_COLS, params))

        # Iterate through each row (paper_id) within the current parameter group
        for _, row in tqdm(group_df.iterrows(), total=len(group_df), desc=f"  Papers in Group {group_idx+1}"):
            global_item_counter += 1
            paper_id = row['paper_id'] # ID of the paper

            # Get formatted parameter string for printing for the current paper-parameter combo
            param_dict_for_print = {col: row[col] for col in PARAM_COLS}
            _, print_params_str = format_param_strings(paper_id, param_dict_for_print)

            current_combo_key = get_unique_combo_key(row, PARAM_COLS)

            if current_combo_key in done_combinations:
                continue # Skip to the next paper_id in this group

            # Call the function to process each paper_id
            results_dict, log_dict = process_single_paper(
                llm_client, MODEL, row['paper_id'], row, response_a_col_name,
                response_b_col_name, merged_df, PARAM_COLS
            )

            all_results_data.append(results_dict)
            all_log_data.append(log_dict)

            # Print formatted messages based on completion status
            if log_dict['complete']:
                print(f"[COMPLETE] paper_id={paper_id} | {print_params_str}; Processing was complete.", flush=True)
            else:
                print(f"[INCOMPLETE] paper_id={paper_id} | {print_params_str}; Processing was incomplete.", flush=True)

    # After processing all papers, save all collected data to Parquet files
    if all_results_data:
        new_results_df = pd.DataFrame(all_results_data, columns=RES_COL)
        if os.path.exists(result_path):
            # If results file already exists, load it, concatenate, and save
            existing_results_df = pd.read_parquet(result_path)
            final_results_df = pd.concat([existing_results_df, new_results_df], ignore_index=True)
            print(f"\nAppended new results to existing file: {result_path}", flush=True)
        else:
            final_results_df = new_results_df
            print(f"\nCreated new results file: {result_path}", flush=True)
        final_results_df.to_parquet(result_path, index=False)
    else:
        print(f"\nNo new results data to save to {result_path}.", flush=True)

    if all_log_data:
        new_log_df = pd.DataFrame(all_log_data, columns=LOG_COL)
        if not existing_log_df.empty: # Check if existing log was actually loaded
            # If log file already exists, load it, concatenate, and save
            final_log_df = pd.concat([existing_log_df, new_log_df], ignore_index=True)
            print(f"Appended new log entries to existing file: {log_path}", flush=True)
        else:
            final_log_df = new_log_df
            print(f"Created new log file: {log_path}", flush=True)
        final_log_df.to_parquet(log_path, index=False)
    else:
        print(f"No new log data to save to {log_path}.", flush=True)

Prompt input will use columns parsed_text and llm_review_1
Processing and saving results to: /content/drive/MyDrive/Thesis/tvd_mi/tvd_mi_scores_paper_llm_review_1.parquet
Logging successful iterations to: /content/drive/MyDrive/Thesis/tvd_mi/tvd_mi_log_paper_llm_review_1.parquet
Starting processing for 5 parameter configuration groups.


Parameter Groups:   0%|          | 0/5 [00:00<?, ?it/s]

  Papers in Group 1:   0%|          | 0/98 [00:00<?, ?it/s]

[INCOMPLETE] paper_id=-Y34L45JR6z | model=gpt-5-mini-2025-08-07, temperature=, reasoning_effort=high, chain_of_thought=, scope=abstract; Processing was incomplete.
[COMPLETE] paper_id=0WVNuEnqVu | model=gpt-5-mini-2025-08-07, temperature=, reasoning_effort=high, chain_of_thought=, scope=abstract; Processing was complete.
[INCOMPLETE] paper_id=1Wo0vqaZ8WJ | model=gpt-5-mini-2025-08-07, temperature=, reasoning_effort=high, chain_of_thought=, scope=abstract; Processing was incomplete.
[INCOMPLETE] paper_id=1usJZBGNrZ | model=gpt-5-mini-2025-08-07, temperature=, reasoning_effort=high, chain_of_thought=, scope=abstract; Processing was incomplete.
[INCOMPLETE] paper_id=2SV2dlfBuE3 | model=gpt-5-mini-2025-08-07, temperature=, reasoning_effort=high, chain_of_thought=, scope=abstract; Processing was incomplete.
[INCOMPLETE] paper_id=3i9EgUss-Vs | model=gpt-5-mini-2025-08-07, temperature=, reasoning_effort=high, chain_of_thought=, scope=abstract; Processing was incomplete.
[INCOMPLETE] paper_id=

  Papers in Group 2:   0%|          | 0/98 [00:00<?, ?it/s]

...

[INCOMPLETE] paper_id=-Y34L45JR6z | model=gpt-5-mini-2025-08-07, temperature=, reasoning_effort=low, chain_of_thought=, scope=full_text; Processing was incomplete.
[INCOMPLETE] paper_id=0WVNuEnqVu | model=gpt-5-mini-2025-08-07, temperature=, reasoning_effort=low, chain_of_thought=, scope=full_text; Processing was incomplete.
[INCOMPLETE] paper_id=1Wo0vqaZ8WJ | model=gpt-5-mini-2025-08-07, temperature=, reasoning_effort=low, chain_of_thought=, scope=full_text; Processing was incomplete.
[INCOMPLETE] paper_id=1usJZBGNrZ | model=gpt-5-mini-2025-08-07, temperature=, reasoning_effort=low, chain_of_thought=, scope=full_text; Processing was incomplete.
[INCOMPLETE] paper_id=2SV2dlfBuE3 | model=gpt-5-mini-2025-08-07, temperature=, reasoning_effort=low, chain_of_thought=, scope=full_text; Processing was incomplete.
[INCOMPLETE] paper_id=3i9EgUss-Vs | model=gpt-5-mini-2025-08-07, temperature=, reasoning_effort=low, chain_of_thought=, scope=full_text; Processing was incomplete.
[INCOMPLETE] paper

  Papers in Group 4:   0%|          | 0/98 [00:00<?, ?it/s]

[INCOMPLETE] paper_id=-Y34L45JR6z | model=gpt-5-mini-2025-08-07, temperature=, reasoning_effort=low, chain_of_thought=Explain your thought process step by step., scope=abstract; Processing was incomplete.
[COMPLETE] paper_id=0WVNuEnqVu | model=gpt-5-mini-2025-08-07, temperature=, reasoning_effort=low, chain_of_thought=Explain your thought process step by step., scope=abstract; Processing was complete.
[INCOMPLETE] paper_id=1Wo0vqaZ8WJ | model=gpt-5-mini-2025-08-07, temperature=, reasoning_effort=low, chain_of_thought=Explain your thought process step by step., scope=abstract; Processing was incomplete.
[INCOMPLETE] paper_id=1usJZBGNrZ | model=gpt-5-mini-2025-08-07, temperature=, reasoning_effort=low, chain_of_thought=Explain your thought process step by step., scope=abstract; Processing was incomplete.
[COMPLETE] paper_id=2SV2dlfBuE3 | model=gpt-5-mini-2025-08-07, temperature=, reasoning_effort=low, chain_of_thought=Explain your thought process step by step., scope=abstract; Processing

  Papers in Group 5:   0%|          | 0/98 [00:00<?, ?it/s]

[INCOMPLETE] paper_id=-Y34L45JR6z | model=gpt-5-mini-2025-08-07, temperature=, reasoning_effort=low, chain_of_thought=Explain your thought process step by step., scope=full_text; Processing was incomplete.
[COMPLETE] paper_id=0WVNuEnqVu | model=gpt-5-mini-2025-08-07, temperature=, reasoning_effort=low, chain_of_thought=Explain your thought process step by step., scope=full_text; Processing was complete.
[INCOMPLETE] paper_id=1Wo0vqaZ8WJ | model=gpt-5-mini-2025-08-07, temperature=, reasoning_effort=low, chain_of_thought=Explain your thought process step by step., scope=full_text; Processing was incomplete.
[INCOMPLETE] paper_id=1usJZBGNrZ | model=gpt-5-mini-2025-08-07, temperature=, reasoning_effort=low, chain_of_thought=Explain your thought process step by step., scope=full_text; Processing was incomplete.
[COMPLETE] paper_id=2SV2dlfBuE3 | model=gpt-5-mini-2025-08-07, temperature=, reasoning_effort=low, chain_of_thought=Explain your thought process step by step., scope=full_text; Proce

In [14]:
# result.parquet-file
tvd_mi_results_df = pd.read_parquet(result_path)

# Check results
display(tvd_mi_results_df.head(5))

# log.parquet-file
tvd_mi_log_df = pd.read_parquet(log_path)

# Check logs
display(tvd_mi_log_df.head(5))

,paper_id,model,temperature,reasoning_effort,chain_of_thought,scope,tpr_response_a,tpr_response_b,tpr_tvd_mi_score_1,tpr_tvd_mi_score_2,tpr_avg_tvd_mi_score,fpr_response_a,fpr_response_b,fpr_tvd_mi_score_1,fpr_tvd_mi_score_2,fpr_avg_tvd_mi_score
0,-Y34L45JR6z,gpt-5-mini-2025-08-07,None,high,,abstract,POLICY EXPANSION FOR BRIDGING OFFLINE-TO-ONLIN...,"{\n ""model"": ""gpt-5-mini-2025-08-07"",\n ""tem...",0.00,0.25,0.125,SIMPLE EMERGENT ACTION REPRESENTATIONS FROM MU...,"{\n ""model"": ""gpt-5-mini-2025-08-07"",\n ""tem...",NaN,1.00,NaN
1,0WVNuEnqVu,gpt-5-mini-2025-08-07,None,high,,abstract,DEEP REINFORCEMENT LEARNING FOR COST-EFFECTIVE...,"{\n ""model"": ""gpt-5-mini-2025-08-07"",\n ""tem...",1.00,1.00,1.000,TOWARDS GLOBAL OPTIMALITY IN COOPERATIVE MARL ...,"{\n ""model"": ""gpt-5-mini-2025-08-07"",\n ""tem...",0.00,1.00,0.500
2,1Wo0vqaZ8WJ,gpt-5-mini-2025-08-07,None,high,,abstract,LET OFFLINE RL FLOW: TRAINING CONSERVATIVE AGE...,"{\n ""model"": ""gpt-5-mini-2025-08-07"",\n ""tem...",0.25,1.00,0.625,GRAPH CONVOLUTIONAL NORMALIZING FLOWS FOR SEMI...,"{\n ""model"": ""gpt-5-mini-2025-08-07"",\n ""tem...",1.00,NaN,NaN
3,1usJZBGNrZ,gpt-5-mini-2025-08-07,None,high,,abstract,OFFLINE REINFORCEMENT LEARNING WITH CLOSED-FOR...,"{\n ""model"": ""gpt-5-mini-2025-08-07"",\n ""tem...",NaN,1.00,NaN,DIRICHLET-BASED UNCERTAINTY CALIBRATION FOR AC...,"{\n ""model"": ""gpt-5-mini-2025-08-07"",\n ""tem...",0.25,1.00,0.625
4,2SV2dlfBuE3,gpt-5-mini-2025-08-07,None,high,,abstract,PREDICTOR-CORRECTOR ALGORITHMS FOR STOCHAS-TIC...,"{\n ""model"": ""gpt-5-mini-2025-08-07"",\n ""tem...",NaN,1.00,NaN,AUTOENCODING HYPERBOLIC REPRESENTATION FOR ADV...,"{\n ""model"": ""gpt-5-mini-2025-08-07"",\n ""tem...",1.00,0.25,0.625


,combo,TPR_calculated,FPR_calculated,complete
0,paper_id=-Y34L45JR6z|model=gpt-5-mini-2025-08-...,True,False,False
1,paper_id=0WVNuEnqVu|model=gpt-5-mini-2025-08-0...,True,True,True
2,paper_id=1Wo0vqaZ8WJ|model=gpt-5-mini-2025-08-...,True,False,False
3,paper_id=1usJZBGNrZ|model=gpt-5-mini-2025-08-0...,False,True,False
4,paper_id=2SV2dlfBuE3|model=gpt-5-mini-2025-08-...,False,True,False


### References

The main code logic and in parts exact code has been taken from:

Robertson, Z., & Koyejo, S. (2025). Let's Measure Information Step-by-Step: LLM-Based Evaluation Beyond Vibes. *arXiv preprint arXiv:2508.05469.*

The code can be accessed [here](https://github.com/zrobertson466920/llm-peer-prediction/tree/main).